# Ejercicio adicional de fin de semana: semana 2

Ahora usa todo lo que aprendiste en la semana 2 para construir un prototipo completo para la pregunta/respuesta técnica que creaste en el ejercicio de la semana 1.

Esto debería incluir una interfaz de usuario de Gradio, transmisión, uso del mensaje del sistema para agregar experiencia y la capacidad de cambiar entre modelos. ¡Puntos extra si puedes demostrar el uso de una herramienta!

Si te sientes audaz, ve si puedes agregar una entrada de audio para poder hablarle y hacer que responda con audio. ChatGPT o Claude pueden ayudarte, o envíame un correo electrónico si tienes preguntas.

Pronto publicaré una solución completa aquí, a menos que alguien se me adelante...

Hay tantas aplicaciones comerciales para esto, desde un tutor de idiomas hasta una solución de incorporación de empresas, pasando por una IA complementaria para un curso (¡como este!). No puedo esperar a ver tus resultados.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from typing import List, Dict
import random

In [2]:
# Inicialización

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
  print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
  print("OpenAI API Key sin configurar")

MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [5]:
def get_tools() -> List[Dict]:
  random_word_function = {
      "name":
          "get_random_english_word",
      "description":
          "Si hay una palabra inventada en español, utiliza esta función para devolver una palabra aleatoria en inglés y así ayudar a traducir la palabra inventada. No uses esta función para nada más que para traducir palabras inventadas.",
      "parameters":
          {
              "type": "object",
              "properties":
                  {
                      "invented_word":
                          {
                              "type":
                                  "string",
                              "description":
                                  "La palabra inventada en español que se desea traducir al inglés",
                          },
                  },
              "required": ["invented_word"],
              "additionalProperties": False
          }
  }
  return [{"type": "function", "function": random_word_function}]


def get_system_message():
  return '''
    Eres un asistente que traduce del español al inglés. Responde solo la traducción sin ningún texto adicional.
  '''


def chat(history):
  messages = [{"role": "system", "content": get_system_message()}] + history
  tools = get_tools()
  response = openai.chat.completions.create(
      model=MODEL, messages=messages, tools=tools
  )
  image = None

  if response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    response = handle_tool_call(message)
    messages.append(message)
    messages.append(response)
    response = openai.chat.completions.create(model=MODEL, messages=messages)

  reply = response.choices[0].message.content
  # reply = response
  history += [{"role": "assistant", "content": reply}]

  return history, image


def get_random_english_word():
  english_words = [
      "serendipity",
      "ephemeral",
      "luminous",
      "quintessential",
      "mellifluous",
  ]
  return random.choice(english_words)


def handle_tool_call(message):
  tool_call = message.tool_calls[0]
  print('tool_call', tool_call)
  # arguments = json.loads(tool_call.function.arguments)
  # city = arguments.get('destination_city')
  # price = get_ticket_price(city)
  response = {
      "role": "tool",
      "content": json.dumps({
          "invented_word": tool_call.function.arguments,
          "english_word": get_random_english_word()
      }),
      "tool_call_id": message.tool_calls[0].id
  }
  return response


if __name__ == "__main__":
  with gr.Blocks() as ui:
    with gr.Row():
      chatbot = gr.Chatbot(height=250)
    with gr.Row():
      entry = gr.Textbox(label="Traduce al ingles.")
    with gr.Row():
      clear = gr.Button("Clear")

    def do_entry(message, history):
      history += [{"role": "user", "content": message}]
      return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]) \
        .then(chat, inputs=chatbot, outputs=[chatbot])

    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

  ui.launch(inbrowser=False, height=500)



* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
